# Name That LLM: a fine-tuned Qwen3-8B, first run

**Goal:** fine-tune Qwen 3 8B (base) to name which of the 18 LLMs wrote a passage, and
measure it the honest way, on seed-7 folds, next to our current models.

Part of the learning-in-public MLOps project
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P7 (Task 5).

**How an 8-billion-weight model trains on Kaggle's two T4s (16 GB each)**
(`name_that_llm/llm.py` explains each in detail):
- its weights load in **4 bits** (NF4) and stay frozen;
- **LoRA** trains small low-rank updates beside every linear layer, plus a new 18-way head;
- **gradient checkpointing** recomputes activations instead of storing them all;
- **two folds at once,** one process per GPU.

**How we keep it honest:**
- prompt-grouped folds, as everywhere in the project: a fold's model never sees its
  validation prompts;
- **seed 7 only:** choices are made on seed 7, and fresh seeds report later (Task 7);
- **pool-free:** our 2,699 passages only. The generated pool is Task 6's question;
- **a timing probe first:** if the full CV would take more than 6 hours, it's skipped.

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
# peft (LoRA) and bitsandbytes (the 4-bit weights) aren't in Kaggle's image.
%pip install -q peft bitsandbytes "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@e97e3b364d8c31a97319186ba725fd6079ff9779"

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import peft
import torch
import transformers

from name_that_llm.blend import best_blend_weights, blend
from name_that_llm.config import BlendConfig, EnsembleConfig, LLMConfig, load_config
from name_that_llm.data import LABELS, load_train
from name_that_llm.llm import merge_folds
from name_that_llm.lm import find_model_dir, select_layers
from name_that_llm.metrics import score
from name_that_llm.train import run_cv, run_cv_features

# Kaggle's folders by default. The smoke test (tests/test_notebooks.py) runs this very
# code on a CPU, with these pointed at a tiny random Qwen and fake data.
INPUT = Path(os.environ.get("NTL_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("NTL_WORK", "/kaggle/working"))
CONFIG = os.environ.get("NTL_LLM_CONFIG", "llm")  # a config name, or a YAML path
ON_KAGGLE = "NTL_INPUT" not in os.environ
SEED = 7  # choices are made on seed 7; fresh seeds only report (Task 7)

try:
    import bitsandbytes

    BNB = bitsandbytes.__version__
except ImportError:
    BNB = "not installed"
print(
    f"torch {torch.__version__} (CUDA {torch.version.cuda}) | transformers "
    f"{transformers.__version__} | peft {peft.__version__} | bitsandbytes {BNB}"
)
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS)
if ON_KAGGLE:
    # Two folds run at once, one per GPU: the T4 x2 accelerator is required.
    assert len(GPUS) == 2, "this notebook needs the T4 x2 accelerator"

# Tracking is optional: without the Kaggle Secret, everything still runs and is saved.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    import wandb

    wandb.login(verify=True)  # checks the key with W&B's server; the key is never printed
    TRACK = True
    print("W&B tracking is ON")
except Exception as exc:
    TRACK = False
    print(f"W&B tracking is OFF ({type(exc).__name__}: {str(exc)[:150]})")
    print("Results are still saved as files.")

train = load_train()
y = train["model"].to_numpy()
labels = np.asarray(LABELS)
BASE = find_model_dir("8b-base", INPUT)  # Kaggle Models' Qwen 3 8B base, wherever mounted
CFG = load_config(CONFIG, model_cls=LLMConfig).model_copy(
    update={"base_model": str(BASE), "seed": SEED}
)
print(CFG)

## 1. The timing probe

One fold's training, cut to 256 passages and 1 epoch, on GPU 0. Its speed gives the
projected time of the full 5-fold CV, where the busier GPU runs 3 folds. Over 6 hours, the
CV is skipped, and Task 6 tries cheaper settings (fewer blocks, shorter passages).

The probe runs in **a process of its own**, like the folds, so all of its GPU memory is
returned when it ends. The first run kept the probe in this notebook's process: 5.8 GB
of the freed model stayed on GPU 0, and the fold that ran there next ran out of memory.

In [ ]:
def command(*args: str) -> list[str]:
    """`python -m name_that_llm.llm` with this run's config, base model and seed."""
    base = ["--config", CONFIG, "--base-model", str(BASE), "--seed", str(SEED)]
    return [sys.executable, "-m", "name_that_llm.llm", *base, *args]


def on_gpu(gpu: int) -> dict:
    """The environment of a process that sees only `gpu` (as cuda:0) and logs in UTF-8."""
    return {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONIOENCODING": "utf-8"}


probe = subprocess.run(
    command("--probe", "256", "--out", str(WORK / "timing.json")),
    env=on_gpu(0),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
if probe.returncode != 0:
    print(probe.stdout[-2000:], probe.stderr[-3000:], sep="\n")
    raise RuntimeError(f"the timing probe failed (exit code {probe.returncode})")
timing = json.loads((WORK / "timing.json").read_text(encoding="utf-8"))
print(timing)
RUN_CV = timing["projected_cv_minutes"] <= 6 * 60
if not RUN_CV:
    minutes = timing["projected_cv_minutes"]
    print(f"The CV is skipped: {minutes:.0f} minutes projected, over the 6-hour limit.")

## 2. The seed-7 CV, two folds at once

Two `python -m name_that_llm.llm` processes, one per GPU: folds 0, 2 and 4 on GPU 0, and
folds 1 and 3 on GPU 1. Every fold is seeded by its own number, so this split gives
exactly the numbers of a single run (`tests/test_llm.py` checks it). Each fold's line is
printed when it finishes.

In [ ]:
def launch(gpu: int, fold_ids: list[int], log) -> subprocess.Popen:
    """Start one CLI process on one GPU, its output going to the open file `log`."""
    args = ["--folds", *map(str, fold_ids), "--out", str(WORK / f"llm-gpu{gpu}.npz")]
    return subprocess.Popen(command(*args), env=on_gpu(gpu), stdout=log, stderr=subprocess.STDOUT)


def log_text(gpu: int) -> str:
    return (WORK / f"llm-gpu{gpu}.log").read_text(encoding="utf-8", errors="replace")


def fold_lines(gpu: int) -> list[str]:
    """The CLI's one line per finished fold, from its log."""
    return [line for line in log_text(gpu).splitlines() if " fold " in line and ": CV " in line]


if RUN_CV:
    start = time.time()
    with open(WORK / "llm-gpu0.log", "w") as log0, open(WORK / "llm-gpu1.log", "w") as log1:
        procs = [launch(0, [0, 2, 4], log0), launch(1, [1, 3], log1)]
        shown = [0, 0]
        while True:
            codes = [p.poll() for p in procs]  # read BEFORE the logs, so no line is missed
            for gpu in (0, 1):
                lines = fold_lines(gpu)
                for line in lines[shown[gpu] :]:
                    print(f"GPU {gpu} | {line}", flush=True)
                shown[gpu] = len(lines)
            failed = [gpu for gpu, code in enumerate(codes) if code not in (None, 0)]
            if failed or None not in codes:
                break
            time.sleep(10)
        for p in procs:
            if p.poll() is None:
                # The other process failed, so the folds can't all be merged: stop spending
                # GPU time on this one.
                p.kill()
                p.wait()
    if failed:
        print(log_text(failed[0])[-3000:])  # the end of its log: usually the traceback
        raise RuntimeError(f"the process on GPU {failed[0]} failed (exit {codes[failed[0]]})")
    parts = []
    for gpu in (0, 1):
        with np.load(WORK / f"llm-gpu{gpu}.npz") as npz:
            parts.append({int(name.removeprefix("fold")): npz[name] for name in npz.files})
    LLM = merge_folds(CFG, train, parts)  # every fold exactly once, or it raises
    np.save(WORK / f"oof-llm-seed{SEED}.npy", LLM.oof.astype(np.float32))
    print(
        f"==> LLM seed {SEED}: CV {LLM.score:.4f} | per fold "
        + ", ".join(f"{s:.4f}" for s in LLM.fold_scores)
        + f" | {(time.time() - start) / 60:.0f} min"
    )

## 3. The comparison, on seed 7

Every model's out-of-fold score on the same seed-7 folds: `lr`, the LM head, the encoder,
the current ensemble, and the LLM. Then the best weights over the three current members,
and over all four. Those weights are chosen and scored on the same folds, so they're a
guide to what Task 7 may find, not a promotion. Task 7 chooses on seed 7 and judges on
fresh folds.

In [ ]:
BLEND = load_config("lm_blend", model_cls=BlendConfig)
ENS = load_config("ensemble", model_cls=EnsembleConfig)
TEXT = load_config(BLEND.text_config)


def find_one(pattern: str) -> Path:
    """The single file matching `pattern` under INPUT (the attached notebooks' outputs)."""
    found = sorted(INPUT.rglob(pattern))
    assert len(found) == 1, f"expected one {pattern}, found {found}"
    return found[0]


# The LM head's features come from the lm-features notebook. They're matched to the labels
# by ROW POSITION, so the row order is checked first (P3b Review Focus 1).
ids = np.load(find_one("train_ids.npy"), allow_pickle=True)
assert (ids == train["id"].to_numpy()).all(), "row order differs!"
features = np.load(find_one(f"features-{BLEND.lm_variation}.npy"))
X_LM = select_layers(features, BLEND.layer, BLEND.layer_window)
del features
oof = {
    "lr": run_cv(TEXT.model_copy(update={"seed": SEED}), train).oof,
    "LM head": run_cv_features(X_LM, train, C=BLEND.C, seed=SEED).oof,
    # The encoder's seed-7 OOF, saved by the encoder-final notebook (same row order).
    "encoder": np.load(find_one(f"oof-encoder-seed{SEED}.npy")),
}
oof["ensemble"] = blend([oof["lr"], oof["LM head"], oof["encoder"]], ENS.weights)
if RUN_CV:
    oof["LLM"] = LLM.oof
scores = {name: score(y, labels[p.argmax(axis=1)]) for name, p in oof.items()}
members = [oof["lr"], oof["LM head"], oof["encoder"]]
w3, scores["best 3-member blend"] = best_blend_weights(members, y)
w4 = None
if RUN_CV:
    # The newcomer goes LAST: on a tie, it gets the smaller weight.
    w4, scores["best 4-member blend"] = best_blend_weights(members + [oof["LLM"]], y)
for name, s in scores.items():
    print(f"{name:22s} {s:.4f}")
print("best weights, lr / LM head / encoder:", w3)
if RUN_CV:
    print("best weights, lr / LM head / encoder / LLM:", w4)
    # Task 5's rule: go on to Task 6 if the LLM alone matches the encoder alone, or if it
    # lifts the blend by at least 0.01. Otherwise, stop: that's a finding.
    gain = scores["best 4-member blend"] - scores["best 3-member blend"]
    go = scores["LLM"] >= scores["encoder"] or gain >= 0.01
    print(f"blend gain {gain:+.4f} -> " + ("continue to Task 6" if go else "stop and ask"))
results = {"seed": SEED, "scores": scores, "weights3": w3, "weights4": w4, "timing": timing}
(WORK / "results.json").write_text(json.dumps(results, indent=2))
if TRACK and RUN_CV:
    from name_that_llm.tracking import log_cv_run

    extra = {f"seed7_{name.replace(' ', '_')}": s for name, s in scores.items()}
    print("W&B run:", log_cv_run(CFG, LLM, train, extra=extra | timing))

## Findings

Run on 2026-10-28 (version 3), Kaggle T4 x2, about 4 h 11 min; pool-free, seed 7, the shipped
`llm` config (all 36 blocks, 512 tokens, LoRA r 16, lr 1e-4, 2 epochs).

| model (seed-7 out-of-fold) | balanced accuracy |
|---|---|
| `lr` | 0.3304 |
| the LM head | 0.3864 |
| the ModernBERT encoder | 0.3590 |
| **Qwen3-8B + LoRA, alone** | **0.3641** |
| the current ensemble (3 members) | 0.4293 |
| **the best 4-member blend** | **0.4523** |

- **The LLM alone beats the encoder** (0.3641 against 0.3590), and **adds +0.023 to the
  blend**. Its best weight is 0.25, with `lr` 0.55, the LM head 0.10 and the encoder 0.10.
  Both of Task 5's rules hold, so P7 continues to Task 6.
- **That blend is in-sample**: its weights are chosen and scored on the same seed-7 folds,
  so it's a guide. Task 7 chooses on seed 7 and judges on fresh seeds.
- **The folds:** 0.3852, 0.3616, 0.3852, 0.3519, 0.3370. Training loss went from about
  3.0 to 1.6 between the two epochs, so a third epoch, or more data (the pool), may help:
  that's Task 6's question.
- **Speed:** 1.02 s per training passage, the model loads in 87 s, peak memory 9.6 GB of
  the T4's 15. The probe projected 233 minutes, and the CV took about 237.
- **The first run (version 2) ran out of memory** on GPU 0. The probe ran in this notebook's
  own process, and 5.8 GB of its freed model stayed on the GPU. The probe now runs in a
  process of its own, like the folds.
- **W&B stayed off** ("ConnectionError" reading the Kaggle Secret), as in every pushed P5
  run, even with the secret attached first. Kaggle doesn't hand secrets to runs pushed
  through its API. The results are saved as files instead (`results.json`,
  `oof-llm-seed7.npy`, `timing.json`).